In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=SM5j22uceZQ29fQdlS0157Ry3Ds0Fg&access_type=offline&code_challenge=mO_1-7TKiG6jHtxZptmIEzB_KAOGMURpfMRvZ14XXZk&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [ ]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()


Loading BokehJS ...

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/22 14:42:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 57067)
Traceback (most recent call last):
  File "/Users/yt4/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/yt4/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/Users/yt4/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/yt4/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/accumulators.py", line 295, in handle

In [3]:
path_to_release_folder="gs://open-targets-data-releases/25.06/"


si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

sl_eff=session.spark.read.parquet("gs://genetics-portal-dev-analysis/ss60/gentropy-manuscript/chapters/variant-effect-prediction/25.07/lead_variant_effect")

# TMP 1

In [9]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_new,
    score_column="score_new",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.384773,1.994680e-03,1.120912,1.710746,6063,30517,100,697,500607
1,3+,1.950881,2.974677e-20,1.689422,2.252803,20263,16317,310,487,500607
2,4+,3.254431,4.126710e-43,2.789135,3.797349,32258,4322,555,242,500607


In [15]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_old,
    score_column="score_old",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.391139,1.452796e-03,1.129564,1.713287,6060,30493,103,721,517326
1,3+,1.896815,2.417431e-19,1.647116,2.184368,20247,16306,326,498,517326
2,4+,3.313816,4.980686e-46,2.848234,3.855503,32242,4311,571,253,517326


In [23]:
test=session.spark.read.parquet("./data/test.parquet").filter(f.col("goldStandardSet")==1)
test.count()

1623

In [24]:
test.show(1)

+--------------------+---------------+---------------+--------------------+--------------------+--------------------+------------------+------------------+------------------+---------------------------------+---------------------------------+---------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------+--------------------------------------+---------------------+----------------------------------+---------------+----------------------------+-------------------+--------------------------------+----------+-----------------------+-------+--------------------+--------------+---------------------+---------------------+
|        studyLocusId|         geneId|goldStandardSet|eQtlColocClppMaximum|pQtlColocClppMaximum|sQtlColocClppMaximum|eQtlColocH4Maximum|pQtlColocH4Maximum|sQtlColocH4Maximum|eQtlColocClppMaximumNeighbourhood|pQtlColocClppMaximumNeighbourhood|sQtlColocClppMaximumNeighbourhood|eQtlColo

In [10]:
train=session.spark.read.parquet("./data/train_v3.parquet").filter(f.col("goldStandardSet")==1).select("studyLocusId","geneId")
train.count()

7386

In [11]:
train=train.join(sl.df.select("studyLocusId","studyId"),on="studyLocusId",how="inner")
train.count()

7386

In [12]:
train=train.join(si.df.select("studyId","diseaseIds"),on="studyId",how="inner")
train.count()

7386

In [13]:
train.show(1)

+----------+--------------------+---------------+--------------------+
|   studyId|        studyLocusId|         geneId|          diseaseIds|
+----------+--------------------+---------------+--------------------+
|GCST011991|e1d7297b86821398b...|ENSG00000007171|[EFO_0000676, MON...|
+----------+--------------------+---------------+--------------------+
only showing top 1 row



In [14]:
train_exploded = train.withColumn("diseaseId", f.explode(f.col("diseaseIds")))
train_exploded.count()

10579

In [15]:
train_exploded.show(1)

+----------+--------------------+---------------+--------------------+-----------+
|   studyId|        studyLocusId|         geneId|          diseaseIds|  diseaseId|
+----------+--------------------+---------------+--------------------+-----------+
|GCST011991|e1d7297b86821398b...|ENSG00000007171|[EFO_0000676, MON...|EFO_0000676|
+----------+--------------------+---------------+--------------------+-----------+
only showing top 1 row



In [16]:
chembl_evidence.show(1)

25/08/12 13:33:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+----------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+---------+----------+------------+-----------+--------------+-------------+----+------------------------+-----------------+-----------------------

In [17]:
chembl_evidence.count()

573103

In [18]:
# Remove rows from chembl_evidence where (targetId, diseaseId) matches any (geneId, diseaseId) in train_exploded

# First, make sure train_exploded has columns 'geneId' and 'diseaseId'
to_remove = train_exploded.select(f.col("geneId").alias("targetId"), "diseaseId")

chembl_evidence_filtered = chembl_evidence.join(
    to_remove, on=["targetId", "diseaseId"], how="left_anti"
)
chembl_evidence_filtered.count()

555679

In [19]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_new,
    score_column="score_new",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence_filtered,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

25/08/12 13:34:57 WARN CacheManager: Asked to cache already cached data.        


,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.177612,1.546401e-01,0.946859,1.464599,6059,30335,96,566,500607
1,3+,1.606505,1.749792e-09,1.375940,1.875705,20236,16158,290,372,500607
2,4+,2.891350,2.170313e-28,2.431995,3.437469,32118,4276,478,184,500607


In [37]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_old,
    score_column="score_old",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence_filtered,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

25/08/11 16:04:15 WARN CacheManager: Asked to cache already cached data.        


,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.176302,1.459377e-01,0.948892,1.458212,6057,30323,99,583,517326
1,3+,1.529576,4.723611e-08,1.313537,1.781148,20227,16153,307,375,517326
2,4+,2.782869,6.792345e-27,2.342921,3.305429,32116,4264,498,184,517326


In [20]:
qd_cs=session.spark.read.parquet("gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_credible_sets").select("studyLocusId").cache()
qd_cs.count()

70618

In [21]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_new.join(qd_cs, on="studyLocusId", how="inner"),
    score_column="score_new",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,2.002625,2.925893e-07,1.504024,2.666517,6111,30691,52,523,97952
1,3+,2.136670,6.457771e-19,1.801138,2.534708,20362,16440,211,364,97952
2,4+,3.778932,5.739315e-42,3.169922,4.504945,32432,4370,381,194,97952


In [22]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_new.join(qd_cs, on="studyLocusId", how="inner"),
    score_column="score_new",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence_filtered,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

25/08/12 13:38:22 WARN CacheManager: Asked to cache already cached data.        


,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.681204,4.406613e-04,1.244801,2.270601,6107,30498,48,403,97952
1,3+,1.701362,2.733104e-08,1.409888,2.053094,20335,16270,191,260,97952
2,4+,3.293244,4.728321e-26,2.688799,4.033568,32283,4322,313,138,97952


In [41]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_old.join(qd_cs, on="studyLocusId", how="inner"),
    score_column="score_old",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.927068,5.133370e-07,1.461580,2.540806,6107,30672,56,542,100563
1,3+,2.160326,4.357639e-20,1.826620,2.554997,20355,16424,218,380,100563
2,4+,3.849879,6.463861e-45,3.241544,4.572379,32419,4360,394,204,100563


In [42]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_old.join(qd_cs, on="studyLocusId", how="inner"),
    score_column="score_old",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence_filtered,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

25/08/11 16:07:55 WARN CacheManager: Asked to cache already cached data.        


,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.589868,1.089201e-03,1.189981,2.124135,6104,30493,52,413,100563
1,3+,1.671471,4.485477e-08,1.389366,2.010855,20335,16262,199,266,100563
2,4+,3.062159,2.988093e-23,2.500201,3.750425,32284,4313,330,135,100563


In [44]:
old_l2g=session.spark.read.parquet("gs://open-targets-ppp-data-releases/24.12/output/etl/parquet/evidence/sourceId=ot_genetics_portal")
old_l2g=old_l2g.select("targetId","diseaseId","studyId","score","variantId").withColumnRenamed("score","resourceScore").cache()
old_l2g.count()

781184

In [45]:
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=old_l2g,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.712922,1.131195e-04,1.287199,2.279447,6110,30757,53,457,217543
1,3+,2.013262,1.029277e-14,1.681709,2.410182,20379,16488,194,316,217543
2,4+,3.796779,7.010255e-38,3.152345,4.572954,32476,4391,337,173,217543


In [46]:
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=old_l2g,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence_filtered,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

25/08/11 16:16:48 WARN CacheManager: Asked to cache already cached data.        


,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.394594,2.870720e-02,1.032474,1.883720,6107,30564,49,342,217543
1,3+,1.477543,1.475806e-04,1.209691,1.804704,20355,16316,179,212,217543
2,4+,3.109662,2.440219e-20,2.495963,3.874256,32338,4333,276,115,217543


In [23]:
sl_eff=session.spark.read.parquet("gs://genetics-portal-dev-analysis/ss60/gentropy-manuscript/chapters/variant-effect-prediction/25.07/lead_variant_effect")
sl_eff=sl_eff.drop("leadVariantConsequence").withColumn("absEstimatedBeta",f.abs(f.col("rescaledStatistics.estimatedBeta"))).withColumn("maf",f.col("majorLdPopulationMaf.value"))

In [24]:
qd_sl_eff=sl_eff.join(qd_cs,on="studyLocusId",how="inner").cache()
qd_sl_eff.count()

70618

In [25]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_new.join(qd_sl_eff.filter(f.col("maf")<=0.01).select("studyLocusId"), on="studyLocusId", how="inner"),
    score_column="score_new",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,5.041411,8.490602e-03,1.227008,20.713660,6161,31163,2,51,3498
1,3+,3.773654,1.004354e-05,2.017731,7.057662,20560,16764,13,40,3498
2,4+,6.957185,5.765360e-11,4.056545,11.931932,32786,4538,27,26,3498


In [26]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_new.join(qd_sl_eff.filter(f.col("maf")<0.01).select("studyLocusId"), on="studyLocusId", how="inner"),
    score_column="score_new",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence_filtered,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,6.179916,0.052185,0.843458,45.279510,6154,30870,1,31,3498
1,3+,2.734141,0.007097,1.294367,5.775431,20516,16508,10,22,3498
2,4+,5.699155,0.000007,2.832537,11.466885,32578,4446,18,14,3498


In [ ]:
evidence=chemblDrugEnrichment.to_disease_target_evidence(table_with_score=l2g_new.join(qd_sl_eff.filter(f.col("maf")>=0.01).select("studyLocusId"), on="studyLocusId", how="inner"),
    score_column="score_new",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.5
)
enrich=chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence_filtered,
    indirect_assoc_score_thr=0.5,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.651623,6.500985e-04,1.222556,2.231274,6107,30505,48,396,96320
1,3+,1.717724,2.112057e-08,1.421176,2.076151,20339,16273,187,257,96320
2,4+,3.297188,9.644737e-26,2.688000,4.044436,32288,4324,308,136,96320


25/08/13 11:40:03 WARN TransportChannelHandler: Exception in connection from macbookpro/192.168.0.39:53484
java.io.IOException: Operation timed out
	at java.base/sun.nio.ch.FileDispatcherImpl.read0(Native Method)
	at java.base/sun.nio.ch.SocketDispatcher.read(SocketDispatcher.java:39)
	at java.base/sun.nio.ch.IOUtil.readIntoNativeBuffer(IOUtil.java:276)
	at java.base/sun.nio.ch.IOUtil.read(IOUtil.java:233)
	at java.base/sun.nio.ch.IOUtil.read(IOUtil.java:223)
	at java.base/sun.nio.ch.SocketChannelImpl.read(SocketChannelImpl.java:356)
	at io.netty.buffer.PooledByteBuf.setBytes(PooledByteBuf.java:254)
	at io.netty.buffer.AbstractByteBuf.writeBytes(AbstractByteBuf.java:1132)
	at io.netty.channel.socket.nio.NioSocketChannel.doReadBytes(NioSocketChannel.java:357)
	at io.netty.channel.nio.AbstractNioByteChannel$NioByteUnsafe.read(AbstractNioByteChannel.java:151)
	at io.netty.channel.nio.NioEventLoop.processSelectedKey(NioEventLoop.java:788)
	at io.netty.channel.nio.NioEventLoop.processSelect

In [58]:
se1=(np.log(3.759225)-np.log(2.502136))/(2*1.96)
se1

0.10384389082991022

In [60]:
se2=(np.log(7.692544)-np.log(1.572677))/(2*1.96)
se2

0.40496741968830696

In [61]:
(3.478202-3.066935)/np.sqrt((se1**2+se2**2))

0.983728619162061

25/08/11 19:07:53 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1072824 ms exceeds timeout 120000 ms
25/08/11 19:07:53 WARN SparkContext: Killing executors is not supported by current scheduler.
25/08/11 19:23:39 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

# Preparing the data

In [3]:
l2g_full=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_prioritised_genes_per_CS.parquet")

In [4]:
qd_cs=session.spark.read.parquet("gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_credible_sets").select("studyLocusId").cache()
qd_cs.count()

70618

In [5]:
l2g_full=l2g_full.join(qd_cs,"studyLocusId","inner").cache()
l2g_full.count()

70400

In [6]:
disease_index_path=path_to_release_folder+"output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

platform_chembl_evidence_path=path_to_release_folder+"output/evidence/sourceId=chembl"
chembl_evidence=session.spark.read.parquet(platform_chembl_evidence_path)

In [7]:
si.df = si.df.withColumn(
    "publicationDate",
    f.when(f.col("publicationDate").isNull(), "2024-12-31").otherwise(f.col("publicationDate"))
)

In [8]:
l2g_full.show(1)

+--------------------+---------------+----------------+----------+----------+---+-----------+
|        studyLocusId|         geneId|           score|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|
+--------------------+---------------+----------------+----------+----------+---+-----------+
|250b6168feb462c63...|ENSG00000002016|0.93658447265625|         1|         0|  0|          1|
+--------------------+---------------+----------------+----------+----------+---+-----------+
only showing top 1 row



In [9]:
sl_eff.printSchema()

root
 |-- variantId: string (nullable = true)
 |-- variant: struct (nullable = true)
 |    |-- chromosome: string (nullable = true)
 |    |-- start: integer (nullable = true)
 |    |-- end: integer (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- ref: string (nullable = true)
 |    |-- alt: string (nullable = true)
 |    |-- length: integer (nullable = true)
 |-- studyLocusId: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- geneId: string (nullable = true)
 |-- originalBeta: double (nullable = true)
 |-- originalStandardError: double (nullable = true)
 |-- locusStatistics: struct (nullable = true)
 |    |-- locusSize: integer (nullable = true)
 |    |-- locusLength: integer (nullable = true)
 |    |-- locusStart: integer (nullable = true)
 |    |-- locusEnd: integer (nullable = true)
 |    |-- leadVariantPIP: double (nullable = true)
 |-- finemappingMethod: string (nullable = true)
 |-- isTransQtl: boolean (nullable = true)
 |-- variantEffect: a

In [10]:
sl_eff=sl_eff.drop("leadVariantConsequence").withColumn("absEstimatedBeta",f.abs(f.col("rescaledStatistics.estimatedBeta"))).withColumn("maf",f.col("majorLdPopulationMaf.value"))

In [11]:
sl_eff.show(2)

+----------------+--------------------+--------------------+--------------------+---------------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------------+-------------------+-------------------+
|       variantId|             variant|        studyLocusId|             studyId|         geneId|originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|traitFromSourceMappedIds|   absEstimatedBeta|                maf|
+----------------+--------------------+--------------------+--------------------+---------------+------------+---------------------+--------------------+-----------------+----------+--------------------+-------------

In [12]:
def to_disease_target_evidence_l(
    table_with_score: DataFrame,
    score_column: str,
    datasource_id: str,
    study_locus_eff: DataFrame,
    study_index: StudyIndex,
    min_score: float = 0.0,
    datatype_id: str = "GWAS",
) -> DataFrame:
    """Convert score from table to disease target evidence.

    The table have to cosist a studyLocusId column.

    Args:
        table_with_score (DataFrame): Table with score
        score_column (str): Column name with score
        datasource_id (str): Data source ID
        study_locus (StudyLocus): Study locus dataset
        study_index (StudyIndex): Study index dataset
        min_score (float): Minimum score to keep
        datatype_id (str): Data type ID

    Returns:
        DataFrame: Disease target evidence
    """
    return (
        table_with_score.filter(f.col(score_column) >= min_score)
        .join(
            study_locus_eff.select("studyLocusId", "studyId","absEstimatedBeta","maf"),
            on="studyLocusId",
            how="inner",
        )
        .join(
            study_index.df.select("studyId", "diseaseIds","nSamples","publicationDate"),
            on="studyId",
            how="inner",
        )
        .select(
            f.lit(datatype_id).alias("datatypeId"),
            f.lit(datasource_id).alias("datasourceId"),
            f.col("geneId").alias("targetId"),
            f.explode(f.col("diseaseIds")).alias("diseaseId"),
            f.col(score_column).alias("resourceScore"),
            "studyLocusId",
            "nSamples","publicationDate","eQTL_coloc","pQTL_coloc","VEP","distanceTSS","absEstimatedBeta","maf"
        )
        .withColumn("year", f.col("publicationDate").substr(1, 4).cast("int"))
        .drop("publicationDate")
    )

In [13]:
evidence=to_disease_target_evidence_l(table_with_score=l2g_full,
    score_column="score",
    datasource_id="l2g",
    study_locus_eff=sl_eff,
    study_index=si,
    min_score=0.1
)
evidence.show(1)

+----------+------------+---------------+-------------+------------------+--------------------+--------+----------+----------+---+-----------+-------------------+-------------------+----+
|datatypeId|datasourceId|       targetId|    diseaseId|     resourceScore|        studyLocusId|nSamples|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|   absEstimatedBeta|                maf|year|
+----------+------------+---------------+-------------+------------------+--------------------+--------+----------+----------+---+-----------+-------------------+-------------------+----+
|      GWAS|         l2g|ENSG00000112425|MONDO_0002041|0.4449980854988098|2b80f2e6a4c8c2bb7...|  500348|         0|         0|  0|          1|0.16858626078822775|0.05144694533762058|2024|
+----------+------------+---------------+-------------+------------------+--------------------+--------+----------+----------+---+-----------+-------------------+-------------------+----+
only showing top 1 row



In [14]:
def evidence_to_indirect_assosiations_l(
        disease_target_evidence: DataFrame,
        disease_index_orig: DataFrame,
        efo_to_remove: list[str] | None = None,
    ) -> DataFrame:
        """Convert evidence to indirect associations.

        Args:
            disease_target_evidence (DataFrame): Disease target evidence
            disease_index_orig (DataFrame): The original disease index (not epxloded)
            use_max (bool): Use max score or harmonic sum (harmonic sum is the default)
            efo_to_remove (list[str] | None): List of EFO IDs to remove
        Returns:
            DataFrame: Direct associations
        """
        if efo_to_remove is not None:
            disease_target_evidence = disease_target_evidence.filter(
                ~f.col("diseaseId").isin(efo_to_remove)
            )

        disease_index = disease_index_orig.select(
            f.col("id").alias("diseaseId"),
            f.explode("ancestors").alias("ancestorDiseaseId"),
        )

        disease_index = disease_index.union(
            disease_index_orig.select(
                f.col("id").alias("diseaseId"),
                f.col("id").alias("ancestorDiseaseId"),
            )
        )

        return (
            disease_target_evidence
            .join(disease_index, on="diseaseId", how="inner")
            .groupBy("targetId", "ancestorDiseaseId")
            .agg(
                f.max("resourceScore").alias("indirect_assoc_score"),
                f.max("nSamples").alias("max_nSamples"),
                f.min("year").alias("min_year"),
                f.max("eQTL_coloc").alias("eQTL_coloc"),
                f.max("pQTL_coloc").alias("pQTL_coloc"),
                f.max("VEP").alias("VEP"),
                f.max("distanceTSS").alias("distanceTSS")
            )
            .withColumn("yearsSinceFirstPublication", 2025-f.col("min_year"))
            .select(
                    "targetId", "ancestorDiseaseId", "indirect_assoc_score", "max_nSamples",
                    "yearsSinceFirstPublication", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS"
            )
            .withColumnRenamed("ancestorDiseaseId", "diseaseId")
        )

In [15]:
x=evidence_to_indirect_assosiations_l(
        disease_target_evidence= evidence,
        disease_index_orig= disease_index_orig,
        efo_to_remove= ["MONDO_0045024"]
    )

In [16]:
x.show(1)

+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+
|       targetId|    diseaseId|indirect_assoc_score|max_nSamples|yearsSinceFirstPublication|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|
+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+
|ENSG00000263528|MONDO_0004670|  0.8069770336151123|      208370|                         9|         1|         0|  0|          1|
+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+
only showing top 1 row



In [17]:
def drug_enrichemnt_from_evidence_prep(
    evid: DataFrame,
    disease_index_orig: DataFrame,
    chembl_orig: DataFrame,
    indirect_assoc_score_thr: float = 0.5,
    efo_ancestors_to_remove: list[str] | None = None,
) -> DataFrame:
    """Run chembl drug enrichment from scores.

    Args:
        evid (DataFrame): Evidence table
        disease_index_orig (DataFrame): The original disease index (not epxloded)
        chembl_orig (DataFrame): Chembl evidence
        indirect_assoc_score_thr (float): Minimum score to keep in indirect associations
        efo_ancestors_to_remove (list[str] | None): List of EFO IDs to remove
    Returns:
        pd.DataFrame: Drug enrichment table.
    """
    if efo_ancestors_to_remove is not None:
        efo_to_remove = (
            chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
                disease_index_orig=disease_index_orig,
                efo_ids=efo_ancestors_to_remove,
            )
        )
    else:
        efo_to_remove = None

    chembl = chemblDrugEnrichment.process_chembl_evidence(
        chembl_orig, efo_to_remove
    )

    evid_indirect = evidence_to_indirect_assosiations_l(
        evid,
        disease_index_orig,
        efo_to_remove=efo_to_remove,
    ).cache()


    joined_data = evid_indirect.join(chembl, ["targetId", "diseaseId"], "right")

    return (
        joined_data
        .withColumn(
            "geneticSupport",
            f.when(
                f.col("indirect_assoc_score") >= indirect_assoc_score_thr, 1
            ).otherwise(0),
        )
        .fillna(0)
        .toPandas()
    )

In [18]:
df=drug_enrichemnt_from_evidence_prep(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.1,
    efo_ancestors_to_remove=["MONDO_0045024"]
)
df

,targetId,diseaseId,indirect_assoc_score,max_nSamples,yearsSinceFirstPublication,eQTL_coloc,pQTL_coloc,VEP,distanceTSS,maxClinicalPhase,geneticSupport
0,ENSG00000004779,MONDO_0020121,0.0,0,0,0,0,0,0,3.0,0
1,ENSG00000011677,EFO_0005411,0.0,0,0,0,0,0,0,3.0,0
2,ENSG00000018625,EFO_0000544,0.0,0,0,0,0,0,0,1.0,0
3,ENSG00000022355,EFO_0007634,0.0,0,0,0,0,0,0,2.0,0
4,ENSG00000053918,EFO_0000546,0.0,0,0,0,0,0,0,2.0,0
...,...,...,...,...,...,...,...,...,...,...,...
37372,ENSG00000273079,EFO_0003108,0.0,0,0,0,0,0,0,2.0,0
37373,ENSG00000274286,EFO_0005207,0.0,0,0,0,0,0,0,3.0,0
37374,ENSG00000274286,MONDO_0001330,0.0,0,0,0,0,0,0,3.0,0
37375,ENSG00000274286,Orphanet_1764,0.0,0,0,0,0,0,0,2.0,0


In [19]:
df["outcome"] = (df["maxClinicalPhase"] > 3).astype(int)

In [20]:
df

,targetId,diseaseId,indirect_assoc_score,max_nSamples,yearsSinceFirstPublication,eQTL_coloc,pQTL_coloc,VEP,distanceTSS,maxClinicalPhase,geneticSupport,outcome
0,ENSG00000004779,MONDO_0020121,0.0,0,0,0,0,0,0,3.0,0,0
1,ENSG00000011677,EFO_0005411,0.0,0,0,0,0,0,0,3.0,0,0
2,ENSG00000018625,EFO_0000544,0.0,0,0,0,0,0,0,1.0,0,0
3,ENSG00000022355,EFO_0007634,0.0,0,0,0,0,0,0,2.0,0,0
4,ENSG00000053918,EFO_0000546,0.0,0,0,0,0,0,0,2.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
37372,ENSG00000273079,EFO_0003108,0.0,0,0,0,0,0,0,2.0,0,0
37373,ENSG00000274286,EFO_0005207,0.0,0,0,0,0,0,0,3.0,0,0
37374,ENSG00000274286,MONDO_0001330,0.0,0,0,0,0,0,0,3.0,0,0
37375,ENSG00000274286,Orphanet_1764,0.0,0,0,0,0,0,0,2.0,0,0


In [23]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Fit logistic regression
model = smf.logit("outcome ~ geneticSupport", data=df).fit()

# Show summary
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.368201
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                37377
Model:                          Logit   Df Residuals:                    37375
Method:                           MLE   Df Model:                            1
Date:                Fri, 22 Aug 2025   Pseudo R-squ.:                0.007818
Time:                        01:03:21   Log-Likelihood:                -13762.
converged:                       True   LL-Null:                       -13871.
Covariance Type:            nonrobust   LLR p-value:                 4.319e-49
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -2.0118      0.016   -124.210      0.000      -2.043      -1.980
geneticSupport   

In [24]:
np.exp(1.2861)

3.6186462796465757

In [25]:
df.to_csv("./data/data_for_drug_enrichment.csv", index=False)

In [29]:
df.to_csv("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/data_for_drug_enrichment.csv", index=False)

In [26]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Fit logistic regression
model = smf.logit("outcome ~ geneticSupport+yearsSinceFirstPublication", data=df).fit()

# Show summary
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.368158
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                37377
Model:                          Logit   Df Residuals:                    37374
Method:                           MLE   Df Model:                            2
Date:                Fri, 22 Aug 2025   Pseudo R-squ.:                0.007934
Time:                        01:04:35   Log-Likelihood:                -13761.
converged:                       True   LL-Null:                       -13871.
Covariance Type:            nonrobust   LLR p-value:                 1.614e-48
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     -2.0118      0.016   -124.210      0.000      

In [27]:
np.exp(1.1981)

3.3138146896007488

In [28]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Fit logistic regression
model = smf.logit("outcome ~ geneticSupport+yearsSinceFirstPublication", data=df).fit()

# Show summary
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.368158
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                outcome   No. Observations:                37377
Model:                          Logit   Df Residuals:                    37374
Method:                           MLE   Df Model:                            2
Date:                Fri, 22 Aug 2025   Pseudo R-squ.:                0.007934
Time:                        01:04:38   Log-Likelihood:                -13761.
converged:                       True   LL-Null:                       -13871.
Covariance Type:            nonrobust   LLR p-value:                 1.614e-48
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                     -2.0118      0.016   -124.210      0.000      

# Estiamting theraputic areas

In [4]:
df = session.spark.read.option("header", "true").csv("./data/data_for_drug_enrichment.csv")

In [5]:
df.show(1)

+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+
|       targetId|    diseaseId|indirect_assoc_score|max_nSamples|yearsSinceFirstPublication|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|maxClinicalPhase|geneticSupport|outcome|
+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+
|ENSG00000004779|MONDO_0020121|                 0.0|           0|                         0|         0|         0|  0|          0|             3.0|             0|      0|
+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+
only showing top 1 row



In [6]:
therapy_area_hierarchy = {
    "EFO_0001444": "measurement",
    "MONDO_0045024": "cancer or benign tumor",
    "OTAR_0000018": "genetic, familial or congenital disease",
    "EFO_0005741": "infectious disease",
    "OTAR_0000009": "injury, poisoning or other complication",
    "OTAR_0000014": "pregnancy or perinatal disease",
    "MONDO_0024458": "disorder of visual system",
    "EFO_0000319": "cardiovascular disease",
    "EFO_0009605": "pancreas disease",
    "EFO_0010282": "gastrointestinal disease",
    "OTAR_0000017": "reproductive system or breast disease",
    "EFO_0010285": "integumentary system disease",
    "EFO_0001379": "endocrine system disease",
    "OTAR_0000010": "respiratory or thoracic disease",
    "EFO_0009690": "urinary system disease",
    "OTAR_0000006": "musculoskeletal or connective tissue disease",
    "MONDO_0021205": "disorder of ear",
    "EFO_0000540": "immune system disease",
    "EFO_0005803": "hematologic disease",
    "EFO_0000618": "nervous system disease",
    "MONDO_0002025": "psychiatric disorder",
    "OTAR_0000020": "nutritional or metabolic disease",
    "EFO_0003765": "sign or symptom",  # Not a therapeutic area - is descendant of phenotype
    # "EFO_0000651": "phenotype",
    # "GO_0008150":  "biological process",
    # "EFO_0002571":  "medical procedure",
    # "EFO_0005932": "animal disease",
}

In [7]:
studies = session.spark.read.parquet(
    "gs://open-targets-data-releases/25.06/output/study"
)

In [9]:
import pyspark.sql.types as t

In [18]:
# This udf extracts the FIRST therapeutic area, as per hierarchy list, for each diseaseId
@f.udf(t.StringType())
def get_first_matching_therapeutic_area(therapeutic_areas_list):
    if therapeutic_areas_list is None:
        return None
    for ta in therapy_area_hierarchy:
        if ta in therapeutic_areas_list:
            return ta
    return None


# These lines create a dictionary of diseaseId to primary therapeutic area
efo_ta = (
    session.spark.read.parquet(
        "gs://open-targets-data-releases/25.06/output/disease/disease.parquet"
    )
    .select(
        "id",
        "ancestors",
    )
    .withColumn(
        "primaryTherapeuticArea",
        get_first_matching_therapeutic_area(f.col("ancestors")),
    )
    .withColumn(
        "primaryTherapeuticArea",
        f.when(f.col("primaryTherapeuticArea").isNull(), f.lit("other")).otherwise(
            f.col("primaryTherapeuticArea")
        ),
    )
)
efo_ta_lookup = efo_ta.select("id", "primaryTherapeuticArea").collect()
efo_ta_dict = {row["id"]: row["primaryTherapeuticArea"] for row in efo_ta_lookup}


# This udf takes a diseaseIds arrays and creates an array of mapped therapeutic areas
@f.udf(t.ArrayType(t.StringType()))
def map_efos_to_therapeutic_areas(efo_ids):
    if efo_ids is None:
        return None
    lookup_dict = efo_ta_dict
    mapped_areas = []
    for efo_id in efo_ids:
        mapped_areas.append(lookup_dict.get(efo_id, None))
        mapped_areas = list(set(area for area in mapped_areas if area is not None))
    return mapped_areas

In [11]:
df.show(1)

+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+
|       targetId|    diseaseId|indirect_assoc_score|max_nSamples|yearsSinceFirstPublication|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|maxClinicalPhase|geneticSupport|outcome|
+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+
|ENSG00000004779|MONDO_0020121|                 0.0|           0|                         0|         0|         0|  0|          0|             3.0|             0|      0|
+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+
only showing top 1 row



In [14]:
df = df.withColumn("diseaseId_array", f.array(f.col("diseaseId"))).cache()
df.show(1)

+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+---------------+
|       targetId|    diseaseId|indirect_assoc_score|max_nSamples|yearsSinceFirstPublication|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|maxClinicalPhase|geneticSupport|outcome|diseaseId_array|
+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+---------------+
|ENSG00000004779|MONDO_0020121|                 0.0|           0|                         0|         0|         0|  0|          0|             3.0|             0|      0|[MONDO_0020121]|
+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+---------------+
only showing top 1 row



In [19]:
df_ta = (
    df
    .withColumn(
        "mappedTherapeuticAreas", map_efos_to_therapeutic_areas(f.col("diseaseId_array"))
    )
    .withColumns(
        {
            "cancerOrBenignTumor": f.when(
                f.array_contains("mappedTherapeuticAreas", "MONDO_0045024"), 1
            ).otherwise(0),
            "infectiousDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0005741"), 1
            ).otherwise(0),
            "pregnancyOrPerinatalDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000014"), 1
            ).otherwise(0),
            "disorderOfVisualSystem": f.when(
                f.array_contains("mappedTherapeuticAreas", "MONDO_0024458"), 1
            ).otherwise(0),
            "cardiovascularDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0000319"), 1
            ).otherwise(0),
            "pancreasDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0009605"), 1
            ).otherwise(0),
            "gastrointestinalDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0010282"), 1
            ).otherwise(0),
            "reproductiveSystemOrBreastDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000017"), 1
            ).otherwise(0),
            "integumentarySystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0010285"), 1
            ).otherwise(0),
            "endocrineSystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0001379"), 1
            ).otherwise(0),
            "respiratoryOrThoracicDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000010"), 1
            ).otherwise(0),
            "urinarySystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0009690"), 1
            ).otherwise(0),
            "musculoskeletalOrConnectiveTissueDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000006"), 1
            ).otherwise(0),
            "disorderOfEar": f.when(
                f.array_contains("mappedTherapeuticAreas", "MONDO_0021205"), 1
            ).otherwise(0),
            "immuneSystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0000540"), 1
            ).otherwise(0),
            "hematologicDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0005803"), 1
            ).otherwise(0),
            "nervousSystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0000618"), 1
            ).otherwise(0),
            "psychiatricDisorder": f.when(
                f.array_contains("mappedTherapeuticAreas", "MONDO_0002025"), 1
            ).otherwise(0),
            "nutritionalOrMetabolicDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000020"), 1
            ).otherwise(0),
            "geneticFamilialOrCongenitalDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000018"), 1
            ).otherwise(0),
            "injuryPoisoningOrOtherComplication": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000009"), 1
            ).otherwise(0),
            "signOrSymptom": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0003765"), 1
            ).otherwise(0),
            "other": f.when(
                f.array_contains("mappedTherapeuticAreas", "other"), 1
            ).otherwise(0),
        }
    )
    .withColumn(
        "totalTherapeuticAreas",
        f.col("cancerOrBenignTumor")
        + f.col("infectiousDisease")
        + f.col("pregnancyOrPerinatalDisease")
        + f.col("disorderOfVisualSystem")
        + f.col("cardiovascularDisease")
        + f.col("pancreasDisease")
        + f.col("gastrointestinalDisease")
        + f.col("reproductiveSystemOrBreastDisease")
        + f.col("integumentarySystemDisease")
        + f.col("endocrineSystemDisease")
        + f.col("respiratoryOrThoracicDisease")
        + f.col("urinarySystemDisease")
        + f.col("musculoskeletalOrConnectiveTissueDisease")
        + f.col("disorderOfEar")
        + f.col("immuneSystemDisease")
        + f.col("hematologicDisease")
        + f.col("nervousSystemDisease")
        + f.col("psychiatricDisorder")
        + f.col("nutritionalOrMetabolicDisease")
        + f.col("geneticFamilialOrCongenitalDisease")
        + f.col("injuryPoisoningOrOtherComplication")
        + f.col("signOrSymptom")
        + f.col("other"),
    )
)

In [20]:
df_ta.show()

+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+---------------+----------------------+-------------------+-----------------+---------------------------+----------------------+---------------------+---------------+-----------------------+---------------------------------+--------------------------+----------------------+----------------------------+--------------------+----------------------------------------+-------------+-------------------+------------------+--------------------+-------------------+-----------------------------+----------------------------------+----------------------------------+-------------+-----+---------------------+
|       targetId|    diseaseId|indirect_assoc_score|max_nSamples|yearsSinceFirstPublication|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|maxClinicalPhase|geneticSupport|outcome|diseaseId_array|mappedTherapeuticAreas|cancerOrBenignTumor

In [21]:
df_ta.groupBy("totalTherapeuticAreas").count().show()

+---------------------+-----+
|totalTherapeuticAreas|count|
+---------------------+-----+
|                    1|37368|
|                    0|    9|
+---------------------+-----+



In [24]:
df_ta.count()

37377

In [25]:
df_ta.toPandas().to_csv("./data/data_for_drug_enrichment_ta.csv", index=False)

In [26]:
studies=si.df.select("diseaseIds","nSamples")

In [28]:
studies.count()

1966178

In [29]:
studies_exploded = studies.withColumn("diseaseId", f.explode(f.col("diseaseIds"))).drop("diseaseIds").cache()
studies_exploded.count()

108620

In [30]:
studies_exploded.show(1)

+--------+-----------+
|nSamples|  diseaseId|
+--------+-----------+
|  373106|EFO_0004237|
+--------+-----------+
only showing top 1 row



In [31]:
studies_aggregated = studies_exploded.groupBy("diseaseId").agg(f.max("nSamples").alias("maxNSamples"))
studies_aggregated.count()

12856

In [32]:
studies_aggregated.show()

+-------------+-----------+
|    diseaseId|maxNSamples|
+-------------+-----------+
|  EFO_0004254|      11561|
|  OBA_2050200|     560575|
|  EFO_0020905|       NULL|
|MONDO_0001493|     616513|
|  OBA_2041341|      46327|
|  EFO_0008167|       1852|
|  OBA_2050107|        984|
|  EFO_0020110|      10708|
|  OBA_2050153|        445|
|  EFO_0020628|      46327|
|  OBA_2040073|       2620|
|  EFO_0803364|      13657|
|  EFO_0800362|      14296|
|  OBA_2044258|      46327|
|  EFO_0800731|       6136|
|  EFO_0011044|       6358|
|  EFO_0802794|      46327|
|  EFO_0020838|      10708|
|   HP_0000031|     564113|
|   HP_0012076|     447600|
+-------------+-----------+
only showing top 20 rows



In [33]:
df_ta.count()

37377

In [34]:
df_ta.show(1)

+---------------+-------------+--------------------+------------+--------------------------+----------+----------+---+-----------+----------------+--------------+-------+---------------+----------------------+-------------------+-----------------+---------------------------+----------------------+---------------------+---------------+-----------------------+---------------------------------+--------------------------+----------------------+----------------------------+--------------------+----------------------------------------+-------------+-------------------+------------------+--------------------+-------------------+-----------------------------+----------------------------------+----------------------------------+-------------+-----+---------------------+
|       targetId|    diseaseId|indirect_assoc_score|max_nSamples|yearsSinceFirstPublication|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|maxClinicalPhase|geneticSupport|outcome|diseaseId_array|mappedTherapeuticAreas|cancerOrBenignTumor

In [35]:
df_ta_n=df_ta.join(studies_aggregated, on="diseaseId", how="left").cache()
df_ta_n.count()

37377

In [37]:
df_ta_n.toPandas().to_csv("./data/data_for_drug_enrichment_ta.csv", index=False)